# Data Exploration

This notebook loads the raw movie metadata, explores the available columns, cleans the fields needed for recommendations, and creates the combined text tags used later for modeling.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import ast
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nojan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nojan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nojan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Load Dataset

Read the raw movie metadata CSV from `data/raw`. The path check supports running the notebook from either the project root or the `notebooks` folder.

In [26]:
from pathlib import Path
import pandas as pd

data_path = Path('data/raw/movies_metadata.csv')
if not data_path.exists():
    data_path = Path('../data/raw/movies_metadata.csv')

if not data_path.exists():
    raise FileNotFoundError("Place movies_metadata.csv in the project's data/raw/ folder.")

df = pd.read_csv(data_path)
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


## Inspect Dataset Structure

Look at the available columns, dataframe shape, and data types to understand what fields are present before cleaning and feature selection.

In [5]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [7]:
df.shape

(45466, 24)

## Check Data Quality

Review missing values and duplicate rows, then drop duplicate records so the model is trained on one copy of each movie row.

In [8]:
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(13)

In [10]:
df = df.drop_duplicates().reset_index(drop=True)

In [11]:
df.duplicated().sum()

np.int64(0)

## Select Recommendation Columns

Keep the movie fields that are useful for content-based recommendations: title, genres, overview, tagline, popularity, collection, and vote average.

In [12]:
df.iloc[0]['belongs_to_collection']

"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}"

In [13]:
movie_df = df[['title', 'genres', 'overview', 'tagline', 'popularity', 'belongs_to_collection', 'vote_average']]

In [14]:
movie_df

,title,genres,overview,tagline,popularity,belongs_to_collection,vote_average
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...",NaN,21.946943,"{'id': 10194, 'name': 'Toy Story Collection', ...",7.7
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,17.015539,NaN,6.9
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,11.7129,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",6.5
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,3.859495,NaN,6.1
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,8.387519,"{'id': 96871, 'name': 'Father of the Bride Col...",5.7
...,...,...,...,...,...,...,...
45448,Subdue,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",Rising and falling between a man and woman.,Rising and falling between a man and woman,0.072051,NaN,4.0
45449,Century of Birthing,"[{'id': 18, 'name': 'Drama'}]",An artist struggles to finish his work while a...,NaN,0.178241,NaN,9.0
45450,Betrayal,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...","When one of her hits goes wrong, a professiona...",A deadly game of wits.,0.903007,NaN,3.8
45451,Satan Triumphant,[],"In a small town live two brothers, one a minis...",NaN,0.003503,NaN,0.0


## Handle Missing Values

Check missing values in the selected columns, remove rows without titles, and replace missing overviews with empty strings so text processing does not fail.

In [15]:
movie_df.isna().sum()

title                        6
genres                       0
overview                   954
tagline                  25045
popularity                   5
belongs_to_collection    40959
vote_average                 6
dtype: int64

In [16]:
movie_df.dropna(inplace=True, axis=0, subset=['title'])

In [17]:
movie_df['overview'] = movie_df['overview'].fillna('')

## Parse Text Metadata

Convert the nested `genres` and `belongs_to_collection` strings into readable text, then fill missing taglines so all text fields can be combined safely.

In [18]:
movie_df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [19]:
movie_df['genres'] = movie_df['genres'].apply(lambda x: " ".join([i['name'] for i in ast.literal_eval(x)]))

In [20]:
movie_df['belongs_to_collection'] = movie_df['belongs_to_collection'].apply(
    lambda x: ast.literal_eval(x)['name'] if isinstance(x, str) else " "
)

In [21]:
movie_df['tagline'] = movie_df['tagline'].fillna('')

In [22]:
movie_df.isna().sum()

title                    0
genres                   0
overview                 0
tagline                  0
popularity               0
belongs_to_collection    0
vote_average             0
dtype: int64

## Build Combined Tags

Create one text field from the cleaned genres, overview, tagline, and collection name. This `tags` column becomes the main text input for the recommendation model.

In [23]:
movie_df['tags'] = movie_df['genres']+ " " + movie_df['overview']+ " " + movie_df['tagline']+ " " + movie_df['belongs_to_collection']

In [24]:
movie_df.iloc[0]['tags']

"Animation Comedy Family Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.  Toy Story Collection"